In [33]:
import torch
from torch import nn

print(torch.__version__)

2.14.0


In [34]:
from pathlib import Path
import pandas as pd

csv_path = next((path for path in (Path("data"), Path("../data")) if (path / "karakalpak-audio-dataset-hf/data.csv").is_file()), None)
if csv_path is None:
    raise FileNotFoundError("karakalpak-audio-dataset-hf/data.csv")

data = pd.read_csv(csv_path / "karakalpak-audio-dataset-hf/data.csv")


In [35]:
data["word_count"] = data["sentence_latin"].fillna("").str.split().str.len()


In [36]:
drop_mask = (data["duration"] < 2) | (data["word_count"] < 4)
print(f"Dropped {drop_mask.sum()} rows; {len(data) - drop_mask.sum()} remain")
data = data.loc[~drop_mask].reset_index(drop=True).copy()


Dropped 419 rows; 2613 remain


In [37]:
data["source_filename"] = data["filename"]
data["audio_index"] = data.index
data["filename"] = [f"audio_{i:06d}.wav" for i in data["audio_index"]]
data = data[["audio_index", "filename", "source_filename", "sentence_latin", "sentence_cyrillic", "duration", "snr_original", "snr_16k", "word_count"]]
data.head()


,audio_index,filename,source_filename,sentence_latin,sentence_cyrillic,duration,snr_original,snr_16k,word_count
0,0,audio_000000.wav,kaa_audio_1000002.wav,Oktyabr ayınıń sońǵı kúnleri.,Октябрь айының соңғы күнлери.,2.967375,36.379130,36.360237,4
1,1,audio_000001.wav,kaa_audio_1000003.wav,"Kún uyasına sıymay dolanıp, qıp-qızıl shoqtay ...","Күн уясына сыймай доланып, қып-қызыл шоқтай жа...",6.952250,25.503586,25.477509,11
2,2,audio_000002.wav,kaa_audio_1000004.wav,“Kese etek”ten Qońıratqa qaray sozılǵan úlken ...,“Кесе етек”тен Қоңыратқа қарай созылған үлкен ...,5.058687,27.380424,27.334051,8
3,3,audio_000003.wav,kaa_audio_1000005.wav,Jeter jerde el joq.,Жетер жерде ел жоқ.,2.006562,30.901669,30.942450,4
4,4,audio_000004.wav,kaa_audio_1000006.wav,"Joldıń eki boyı qula dúz, keń dala (qula dúz -...","Жолдың еки бойы қула дүз, кең дала (қула дүз -...",9.693500,28.682690,28.682480,18


In [38]:
from shutil import copy2

source_dir = csv_path / "karakalpak-audio-dataset-hf"
cleaned_dir = source_dir / "cleaned"
for folder in ("data", "data_16k"):
    (cleaned_dir / folder).mkdir(parents=True, exist_ok=True)


In [39]:
for source_name, new_name in zip(data["source_filename"], data["filename"]):
    for folder in ("data", "data_16k"):
        copy2(source_dir / folder / source_name, cleaned_dir / folder / new_name)
print(f"Copied {len(data)} recordings to each cleaned audio folder")


Copied 2613 recordings to each cleaned audio folder


In [40]:
expected_names = set(data["filename"])
for folder in ("data", "data_16k"):
    for path in (cleaned_dir / folder).glob("*.wav"):
        if path.name not in expected_names:
            path.unlink()


In [41]:
data.to_csv(cleaned_dir / "data.csv", index=False)
print(f"Saved {len(data)} reindexed rows to {cleaned_dir / 'data.csv'}")


Saved 2613 reindexed rows to ../data/karakalpak-audio-dataset-hf/cleaned/data.csv


In [42]:
data["snr_16k"].describe()

count    2613.000000
mean       28.972068
std         2.398034
min        22.414452
25%        27.233220
50%        28.804047
75%        30.508842
max        40.217823
Name: snr_16k, dtype: float64